## Setup

In [ ]:
!git clone https://github.com/realBarry123/audio-embed-experiments
%cd audio-embed-experiments

In [ ]:
!source /venv/main/bin/activate

In [ ]:
!/venv/main/bin/pip install -r requirements.txt
!/venv/main/bin/pip install -e .

In [ ]:
import huggingface_hub
huggingface_hub.notebook_login()

In [ ]:
import wandb
wandb.login()

In [ ]:
!mkdir experiments/sae/results

## Training

In [ ]:
!/venv/main/bin/python experiments/sae/sae.py

In [ ]:
!/venv/main/bin/python experiments/sae/probe.py

In [ ]:
!/venv/main/bin/python experiments/sae/melody_probe.py

## Reloading

In [ ]:
!git pull

In [ ]:
%cd audio-embed-experiments

In [ ]:
import importlib
import audembed
importlib.reload(audembed)
importlib.reload(audembed.audio)
importlib.reload(audembed.data)
importlib.reload(audembed.audio_datasets)
importlib.reload(audembed.models)

In [ ]:
!rm -rf /root/.cache/huggingface/datasets
!rm -rf /root/.local/share/Trash/files

## Testing

### SAE

In [ ]:
!/venv/main/bin/python experiments/sae/test_sae.py

In [ ]:
%matplotlib inline
import torch
import matplotlib.pyplot as plt
from audembed import data

DIMS = 64
latent = torch.load("experiments/sae/results/orchset_sae_latent.pt")
plt.rcParams['figure.figsize'] = [latent.shape[1] * 0.2, DIMS * 0.15]
data.plot_heatmap_2d(
    latent[:DIMS],
    #data.sort_sae_latents(latent), 
    xlabel="t", ylabel="dim", 
    save_file="experiments/sae/results/orchset_sae_latent.png"
)

### Probe

In [ ]:
!/venv/main/bin/python experiments/sae/test_probe.py

In [ ]:
%matplotlib inline
import torch
import matplotlib.pyplot as plt
from audembed import data
state_dict, configs, start_epoch = torch.load("experiments/sae/models/probe.pt", map_location=torch.device('cpu'))

In [ ]:
bins = torch.arange(0, 128, 1)
r2 = torch.load("experiments/sae/results/probe_r2.pt")

plt.scatter(x=bins, y=r2)
plt.xlabel("bins")
plt.ylabel("R^2")
plt.show()

In [ ]:
plt.rcParams['figure.figsize'] = [10, 10]
weight = state_dict["linear.weight"]

from audembed import data
import matplotlib.colors as mcolors
cmap = mcolors.LinearSegmentedColormap.from_list(
    "orange_white_blue", ["orange", "white", "blue"]
)
data.plot_heatmap_2d(
    weight, 
    xlabel="SAE latent", 
    ylabel="bins", 
    cmap=cmap, 
    vmin=-weight.max(), 
    vmax=weight.max()
)


In [ ]:
weight = state_dict["linear.weight"]
plt.plot(weight.norm(dim=1))
plt.xlabel("bins")
plt.ylabel("norm weight")
plt.show()

### Tests

In [ ]:
%matplotlib inline
import torch
from audembed import data, audio, models
from einops import rearrange

state_dict, configs, start_epoch = torch.load("experiments/sae/models/sae.pt")
sae = models.SAE(**configs).to("cuda")
sae.load_state_dict(state_dict)

state_dict, configs, start_epoch = torch.load("experiments/sae/models/probe.pt")
probe = models.LinearProbe(**configs).to("cuda")
probe.load_state_dict(state_dict)

vae = models.VAEWrapper("/", "cuda")

In [ ]:
x = audio.generate_audio(torch.tensor(440).unsqueeze(0).repeat(44100, 1)).unsqueeze(0)
x = vae.encode(x.float().to("cuda"))
with torch.no_grad():
    x = sae.encode(x)
x = probe(x)
x = rearrange(x, "1 frame feat -> feat frame")

data.plot_heatmap_2d(
    x,
    xlabel="t", ylabel="freq", 
    save_file=None
)

## Other Stuff

In [ ]:
!/venv/main/bin/python experiments/sae/reorder_sae.py

In [ ]:
import torch
from audembed import models
state_dict, configs = torch.load("experiments/sae/models/reordered_sae.pt")
print(configs)
sae = models.SAE(**configs).to("cuda")
sae.load_state_dict(state_dict)